# ComfyUI Colab + Vision LLM (vision r1)

ComfyUI + **Vision LLM（画像理解）** 対応版。Colab 上で ComfyUI を操作し、
**画像を入力 → テキストで指示 → プロンプト生成 → img2img** の流れが使えます。

## 主な機能
- ✅ ComfyUI-LLMs-Toolkit（Gemini Vision API 連携）
- ✅ 画像 + 指示文 → LLM がプロンプトを生成
- ✅ 生成プロンプトを Anima img2img に接続可能
- ✅ Colab Secrets で API Key を安全に管理

## 事前準備
1. [Google AI Studio](https://aistudio.google.com/apikey) で Gemini API Key を取得
2. Colab 左パネル 🔑 Secrets に `GOOGLE_API_KEY` を登録

## r2 からの継承機能
- Anima base v1.0 / ComfyUI-Manager / IP-Adapter
- Google Drive 出力 / huggingface_hub モデルDL / xvfb


## Step 1: Environment Setup
ComfyUI のセットアップ、依存パッケージのインストールを行います。

In [ ]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"} Falseにすると100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True   #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

# ── Google Drive 連動（USE_GOOGLE_DRIVE フラグで制御）──
if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive
    print(f"✅ Google Drive モード: 画像は {WORKSPACE}/output に保存されます")
else:
    print(f"✅ ローカルモード: 画像は {WORKSPACE}/output に保存されます")

# ── ComfyUI のクローン / 更新 ──
![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
    !echo -= Updating ComfyUI =-
    ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
    ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
    ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
    ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
    ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
    ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat
    !git pull

# ── 依存パッケージ ──
!echo -= Install dependencies =-
!pip install -q accelerate
!pip install -q einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3

# 【r2改善①】torch は Colab 既存の cu128 版をそのまま使う（再インストール不要）
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  ← 削除
print("ℹ️  torch: Colab 既存ビルドを使用 (再インストールをスキップ)")
import torch; print(f"   torch version: {torch.__version__}")

!pip install -q torchsde
!pip install -q kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install -q comfyui-workflow-templates
!pip install -q comfyui-embedded-docs

# 【r2改善②】nodes_math / nodes_glsl エラーを解消する追加パッケージ
!pip install -q simpleeval
!pip install -q pyopengl

# 【r2改善③】fp8/fp4 量子化対応
!pip install -q comfy-kitchen

# ── ComfyUI-Manager ──
if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd custom_nodes
    ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
    ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
    ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
    ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
    ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull

# ComfyUI-Impact-Pack
![ ! -d $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack ] && git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack

# ComfyUI-LLMs-Toolkit（Vision LLM: Gemini / OpenAI互換API）
INSTALL_VISION_LLM = True  #@param {type:"boolean"} Vision LLM ノードをインストール
if INSTALL_VISION_LLM:
    ![ ! -d $WORKSPACE/custom_nodes/ComfyUI-LLMs-Toolkit ] && \
      git clone https://github.com/ComfyUI-Kelin/ComfyUI-LLMs-Toolkit.git \
      $WORKSPACE/custom_nodes/ComfyUI-LLMs-Toolkit
    !pip install -q -r $WORKSPACE/custom_nodes/ComfyUI-LLMs-Toolkit/requirements.txt
    !pip install -q openai

# ComfyUI_IPAdapter_plus（キャラクター参照画像を使ったimg2imgに使用）
![ ! -d $WORKSPACE/custom_nodes/ComfyUI_IPAdapter_plus ] && git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git $WORKSPACE/custom_nodes/ComfyUI_IPAdapter_plus

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
    !echo -= Install custom nodes dependencies =-
    !pip install -q GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# rembg / onnxruntime / insightface
!pip install -q rembg onnxruntime insightface

# av / comfy_aimdo
!pip install -q av comfy_aimdo

print("\n✅ Step 1 完了（Vision LLM ノード含む）")

## Step 1b: Vision LLM（Gemini）API Key 設定

Colab Secrets に登録した `GOOGLE_API_KEY` を ComfyUI-LLMs-Toolkit に渡します。

| 設定場所 | 値 |
|---------|----|
| Colab Secrets 名 | `GOOGLE_API_KEY` |
| プロバイダー | Gemini（OpenAI 互換 API） |
| モデル | `gemini-2.0-flash`（デフォルト） |


In [ ]:
#@title Gemini API Key 設定（ComfyUI-LLMs-Toolkit 用）

import json, os, shutil
from pathlib import Path

try:
    from google.colab import userdata
    _api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    _api_key = ''

GEMINI_API_KEY = _api_key if _api_key else ''  #@param {type:"string"} Secrets 未設定時はここに直接入力
GEMINI_MODEL   = 'gemini-2.0-flash'  #@param ["gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro"]

_workspace = '/content/ComfyUI'
if 'WORKSPACE' in dir() and WORKSPACE:
    _workspace = WORKSPACE

toolkit_dir = Path(_workspace) / 'custom_nodes' / 'ComfyUI-LLMs-Toolkit'
config_dir  = toolkit_dir / 'config'
config_dir.mkdir(parents=True, exist_ok=True)

default_path = config_dir / 'default_providers.json'
providers_path = config_dir / 'providers.json'

if not default_path.exists():
    raise FileNotFoundError(
        f'ComfyUI-LLMs-Toolkit not found: {toolkit_dir}\nRun Step 1 first.'
    )

if not providers_path.exists():
    shutil.copy(default_path, providers_path)

with open(providers_path, encoding='utf-8') as f:
    cfg = json.load(f)

gemini = {
    'id': 'gemini',
    'name': 'Gemini',
    'type': 'openai',
    'apiKey': GEMINI_API_KEY,
    'apiHost': 'https://generativelanguage.googleapis.com/v1beta/openai',
    'models': ['gemini-2.0-flash', 'gemini-1.5-flash', 'gemini-1.5-pro'],
    'enabled': bool(GEMINI_API_KEY),
    'isSystem': False,
    'skipSSLVerify': True,
}

providers = [p for p in cfg.get('providers', []) if p.get('id') != 'gemini']
providers.append(gemini)
cfg['providers'] = providers

with open(providers_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

if GEMINI_API_KEY:
    print('Gemini provider configured')
    print(f'  Model: {GEMINI_MODEL}')
    print(f'  Config: {providers_path}')
    print('  In ComfyUI: Provider=Gemini, Model=' + GEMINI_MODEL)
else:
    print('WARNING: API Key is empty')
    print('  Set GOOGLE_API_KEY in Colab Secrets or enter it in the param field above')


## Step 2: 仮想ディスプレイ設定（GLSLノード有効化）
Colab はヘッドレス環境のため、OpenGL を必要とする `nodes_glsl.py` を使うには xvfb が必要です。

In [2]:
#@title 【r2改善④】xvfb 仮想ディスプレイのセットアップ（GLSLノード有効化）

!apt-get install -y -q xvfb
import subprocess, os, time

# 既存の :99 プロセスがあれば終了
subprocess.run(["pkill", "-f", "Xvfb :99"], capture_output=True)
time.sleep(0.5)

# 仮想ディスプレイ起動
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1024x768x24"])
os.environ['DISPLAY'] = ':99'
time.sleep(1)

print("✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)")

Reading package lists...
Building dependency tree...
Reading state information...
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)


## Step 3: 出力フォルダの設定
生成画像を常に Google Drive の  へ保存します。


In [3]:
#@title 【r2改善⑤】出力先設定（常に Google Drive へ保存）

import os, shutil
from google.colab import drive

# Google Drive マウント
drive.mount("/content/drive", force_remount=False)

# WORKSPACE を再定義（Step1 未実行でも動くよう独立させる）
_workspace = "/content/ComfyUI"

# Drive 側の出力フォルダを作成
drive_output = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output, exist_ok=True)

# ComfyUI の output をシンボリックリンクで差し替え
local_output = f"{_workspace}/output"
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    shutil.rmtree(local_output)

os.symlink(drive_output, local_output)

# 確認
assert os.path.islink(local_output)
assert os.path.exists(local_output)
print("output ->", os.path.realpath(local_output))
print("setup complete: images will be saved to MyDrive/ComfyUI_Output")


Mounted at /content/drive
output -> /content/drive/MyDrive/ComfyUI_Output
setup complete: images will be saved to MyDrive/ComfyUI_Output


## Step 4: モデルのダウンロード
`huggingface_hub` を使ってリトライ付き・整合性チェック付きでダウンロードします。

| モデルファイル | 用途 | デフォルト |
|---|---|---|
| `anima-base-v1.0.safetensors` | diffusion model（推奨） | ✅ ON |
| `anima-preview.safetensors` | diffusion model（旧プレビュー版） | OFF |
| `qwen_3_06b_base.safetensors` | text encoder（共通） | 常時 |
| `qwen_image_vae.safetensors` | VAE（共通） | 常時 |

In [ ]:
#@title 【r2改善⑥】Anima モデルのダウンロード（base v1.0 対応版）

from huggingface_hub import hf_hub_download
import os, shutil, glob

REPO_ID = "circlestone-labs/Anima"
MODEL_BASE = f"{WORKSPACE}/models"

# ダウンロードするモデルを選択
DOWNLOAD_BASE_V1  = True   #@param {type:"boolean"} base v1.0（推奨）
DOWNLOAD_PREVIEW  = False  #@param {type:"boolean"} preview（旧バージョン）

def download_model(repo_id, hf_filename, local_dir):
    """
    hf_hub_download は filename のサブディレクトリ構造をそのまま再現するため、
    ダウンロード後に目的のフォルダへ移動する。
    """
    os.makedirs(local_dir, exist_ok=True)
    basename = os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# ── diffusion model ──
if DOWNLOAD_BASE_V1:
    download_model(REPO_ID,
        "split_files/diffusion_models/anima-base-v1.0.safetensors",
        f"{MODEL_BASE}/diffusion_models")

if DOWNLOAD_PREVIEW:
    download_model(REPO_ID,
        "split_files/diffusion_models/anima-preview.safetensors",
        f"{MODEL_BASE}/diffusion_models")

# ── 共通モデル（base v1.0 / preview 共通）──
download_model(REPO_ID,
    "split_files/text_encoders/qwen_3_06b_base.safetensors",
    f"{MODEL_BASE}/text_encoders")

download_model(REPO_ID,
    "split_files/vae/qwen_image_vae.safetensors",
    f"{MODEL_BASE}/vae")

# 保存先を確認
print("")
print("saved model files:")
for p in sorted(glob.glob(f"{MODEL_BASE}/**/*.safetensors", recursive=True)):
    print(" ", p)

print("")
print("model download complete")
print("")
print("【base v1.0 推奨パラメータ】")
print("  解像度: 512² ～ 1536²")
print("  Steps : 30-50  /  CFG: 4-5")
print("  Sampler: er_sde（デフォルト）/ euler_a / dpmpp_2m_sde_gpu")

In [5]:
#@title オプション: その他のモデル（コメントアウトを外して使用）

# from huggingface_hub import hf_hub_download
# MODEL_BASE = f"{WORKSPACE}/models"

# ── SDXL ──
# hf_hub_download("stabilityai/stable-diffusion-xl-base-1.0",
#     "sd_xl_base_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)
# hf_hub_download("stabilityai/stable-diffusion-xl-refiner-1.0",
#     "sd_xl_refiner_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)

# ── FLUX.1 ──
# hf_hub_download("black-forest-labs/FLUX.1-schnell",
#     "flux1-schnell.safetensors", local_dir=f"{MODEL_BASE}/diffusion_models", local_dir_use_symlinks=False)

# ── VAE (汎用) ──
# hf_hub_download("stabilityai/sd-vae-ft-mse-original",
#     "vae-ft-mse-840000-ema-pruned.safetensors", local_dir=f"{MODEL_BASE}/vae", local_dir_use_symlinks=False)

# ── UpScale ──
# import urllib.request
# urllib.request.urlretrieve(
#     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
#     f"{MODEL_BASE}/upscale_models/RealESRGAN_x4plus.pth")

print("オプションモデルセルです。必要なものをコメントアウト解除してください。")

オプションモデルセルです。必要なものをコメントアウト解除してください。


In [ ]:
#@title Step 4b: IP-Adapter モデルのダウンロード（ComfyUI_IPAdapter_plus 用）

from huggingface_hub import hf_hub_download
import os, shutil, glob

MODEL_BASE      = f"{WORKSPACE}/models"
CLIP_VISION_DIR = f"{MODEL_BASE}/clip_vision"
IPADAPTER_DIR   = f"{MODEL_BASE}/ipadapter"

# ── ダウンロードするモデルを選択 ──
DOWNLOAD_CLIP_VIT_H    = True   #@param {type:"boolean"} CLIP ViT-H-14（Plus系モデルで必須）
DOWNLOAD_IPA_PLUS_SD15 = True   #@param {type:"boolean"} ip-adapter-plus_sd15（SD1.5用）
DOWNLOAD_IPA_PLUS_SDXL = False  #@param {type:"boolean"} ip-adapter-plus_sdxl_vit-h（SDXL用）

def download_ipa(repo_id, hf_filename, local_dir, save_as=None):
    os.makedirs(local_dir, exist_ok=True)
    basename = save_as if save_as else os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp_ipa"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# ── CLIP Vision ViT-H-14（画像エンコーダ、Plus 系モデルで必須）──
if DOWNLOAD_CLIP_VIT_H:
    download_ipa(
        "h94/IP-Adapter",
        "models/image_encoder/model.safetensors",
        CLIP_VISION_DIR,
        save_as="CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors",
    )

# ── IP-Adapter Plus SD1.5 ──
if DOWNLOAD_IPA_PLUS_SD15:
    download_ipa(
        "h94/IP-Adapter",
        "models/ip-adapter-plus_sd15.safetensors",
        IPADAPTER_DIR,
    )

# ── IP-Adapter Plus SDXL ViT-H ──
if DOWNLOAD_IPA_PLUS_SDXL:
    download_ipa(
        "h94/IP-Adapter",
        "sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors",
        IPADAPTER_DIR,
    )

print("\n保存済みファイル:")
for p in sorted(glob.glob(f"{CLIP_VISION_DIR}/*.safetensors") +
                glob.glob(f"{IPADAPTER_DIR}/*.safetensors")):
    print(f"  {p}")

print("\n✅ IP-Adapter モデルダウンロード完了")
print("")
print("【ComfyUI ワークフローでの設定】")
print("  CLIPVisionLoader     : CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")
print("  IPAdapterModelLoader : ip-adapter-plus_sd15.safetensors")
print("")
print("⚠️  注意: Anima (Wan 系アーキテクチャ) への IPAdapter 適用は実験的です。")
print("   上記モデルは SD1.5 / SDXL 向けのため、Anima では動作しない場合があります。")
print("   Anima 専用の IPAdapter 重みがリリースされた場合は models/ipadapter/ に追加してください。")

In [ ]:
#@title モデル名の互換シンボリックリンク作成（旧ワークフロー対応）
# ワークフロー内で古いモデル名を参照している場合に実行してください。
# 新しいファイルへのエイリアスを作成します（実ファイルはコピーしません）。

import os

DIFFUSION_DIR = f"{WORKSPACE}/models/diffusion_models"

# 旧名 → 新名 のマッピング
ALIASES = {
    "anima-preview.safetensors": "anima-base-v1.0.safetensors",
}

created = []
for alias, real in ALIASES.items():
    alias_path = os.path.join(DIFFUSION_DIR, alias)
    real_path  = os.path.join(DIFFUSION_DIR, real)

    if os.path.exists(alias_path):
        print(f"skip (exists): {alias}")
        continue
    if not os.path.exists(real_path):
        print(f"⚠️  実ファイルが見つかりません: {real}  (Step 4 でダウンロードしてください)")
        continue

    os.symlink(real_path, alias_path)
    created.append(f"  {alias}  →  {real}")

if created:
    print("✅ シンボリックリンクを作成しました:")
    for c in created:
        print(c)
else:
    print("ℹ️  新規リンクなし（すべて処理済みまたはスキップ）")

## Step 4c: Vision LLM クイックテスト（ComfyUI 起動前・任意）

ComfyUI を起動する前に、Gemini Vision が動作するか確認できます。
画像をアップロードし、指示文を入力して実行してください。


In [ ]:
#@title Vision LLM クイックテスト（画像 + 指示 -> テキスト）

!pip install -q google-genai

from google import genai
from google.colab import files, userdata
from IPython.display import display
from PIL import Image
import io

try:
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    api_key = ''

INSTRUCTION = 'この画像のキャラクター特徴を維持しつつ、背景を夜空に変更する Anima 用英語プロンプトを1段落で出力してください。'  #@param {type:"string"}
GEMINI_MODEL  = 'gemini-2.0-flash'  #@param ["gemini-2.0-flash", "gemini-1.5-flash"]

print('Select an image file:')
uploaded = files.upload()
if not uploaded:
    print('No image uploaded')
elif not api_key:
    print('GOOGLE_API_KEY not set. Run Step 1b first.')
else:
    fname = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')
    w = min(512, image.width)
    h = int(image.height * w / image.width)
    display(image.resize((w, h)))

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[INSTRUCTION, image],
    )
    print('\n--- LLM output ---')
    print(response.text)


## Step 4d: Vision ワークフローの配置

サンプルワークフローを ComfyUI から読み込める場所にコピーします。


In [ ]:
#@title Vision + img2img サンプルワークフローの配置

import shutil
from pathlib import Path

_workspace = '/content/ComfyUI'
if 'WORKSPACE' in dir() and WORKSPACE:
    _workspace = WORKSPACE

wf_dir = Path(_workspace) / 'user' / 'default' / 'workflows'
wf_dir.mkdir(parents=True, exist_ok=True)

src_vision = (
    Path(_workspace)
    / 'custom_nodes'
    / 'ComfyUI-LLMs-Toolkit'
    / 'examples'
    / 'workflows'
    / '[LLMs_Toolkit]02_Load_LLMs_ImageLoader.json'
)
if src_vision.exists():
    dst = wf_dir / 'vision_image_instruction.json'
    shutil.copy(src_vision, dst)
    print(f'Vision sample: {dst}')
else:
    print('Vision sample not found. Install ComfyUI-LLMs-Toolkit in Step 1.')

img2img_dst = wf_dir / 'img2img_character_ref2.json'
img2img_copied = False
for candidate in [
    Path('/content/drive/MyDrive/P001/img2img_character_ref2.json'),
    Path('/content/P001/img2img_character_ref2.json'),
    Path('/content/img2img_character_ref2.json'),
]:
    if candidate.exists():
        shutil.copy(candidate, img2img_dst)
        print(f'img2img workflow: {img2img_dst}')
        img2img_copied = True
        break

if not img2img_copied:
    import urllib.request
    raw_url = 'https://raw.githubusercontent.com/WOCae/P001/main/img2img_character_ref2.json'
    try:
        urllib.request.urlretrieve(raw_url, img2img_dst)
        print(f'img2img workflow (from GitHub): {img2img_dst}')
        img2img_copied = True
    except Exception as e:
        print(f'img2img workflow download failed: {e}')

if not img2img_copied:
    print('Upload img2img_character_ref2.json manually if needed')

print('\nUsage in ComfyUI:')
print('1. Start ComfyUI (Step 6) and open the URL')
print('2. Load -> workflows/vision_image_instruction.json')
print('3. Select image in LoadImage node (input/ folder)')
print('4. Enter instruction in OpenAI Compatible Adapter prompt')
print('5. Queue Prompt -> connect output to CLIP Text Encode for img2img')


## Step 5: Vision LLM の使い方（ComfyUI 内）

### 基本フロー

```
LoadImage（参照画像）
    ↓
Image Preprocessor
    ↓
OpenAI Compatible Adapter（Provider: Gemini / Model: gemini-2.0-flash）
    ← prompt に「この画像を〇〇にして」などの指示を入力
    ↓
出力テキスト → CLIP Text Encode → KSampler → SaveImage
```

### 画像のアップロード方法

| 方法 | 手順 |
|------|------|
| **ComfyUI UI** | LoadImage ノード → upload ボタン |
| **input フォルダ** | `/content/ComfyUI/input/` に配置 |
| **Colab セル** | 下のセルで input フォルダへアップロード |

### プロンプト例

- `この画像のキャラクターを維持し、背景を海辺の夕焼けに変更する Anima 用英語プロンプトを出力して`
- `画像の構図を保ちつつ、服装を和服に変更する positive / negative プロンプトを英語で`
- `この画像に写っている要素を箇条書きで説明して`


In [ ]:
#@title 画像を ComfyUI input フォルダへアップロード

from google.colab import files
import os

_workspace = '/content/ComfyUI'
if 'WORKSPACE' in dir() and WORKSPACE:
    _workspace = WORKSPACE

input_dir = f'{_workspace}/input'
os.makedirs(input_dir, exist_ok=True)

print('Select images for ComfyUI:')
uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(input_dir, name)
    with open(path, 'wb') as f:
        f.write(data)
    print(f'Saved: {path}')
if uploaded:
    first = next(iter(uploaded.keys()))
    print(f'\nIn LoadImage node, select: {first}')


## Step 5: 既存ワークフロー（PNG埋め込み）の読み込み

ComfyUI はワークフロー情報を PNG のメタデータに保存します。  
読み込みはブラウザ側で処理されるため、**ファイルを Colab にアップロードする必要はありません**。

### 読み込み方法

| 方法 | 手順 |
|------|------|
| **ドラッグ&ドロップ** | ComfyUI の画面上にローカル PC の PNG をドラッグする |
| **Load ボタン** | 右クリックメニューまたはメニューバーの `Load` からローカルの PNG を選択 |

### モデル名の不一致に注意

ワークフロー内のモデル名と実際にダウンロードされたファイル名が一致しないとエラーになります。  
下のセルで旧モデル名 → 新モデル名のシンボリックリンクを作成できます。

## Step 6: ComfyUI の起動

**cloudflared（推奨）** か **localtunnel** か **Colab iframe** の3種類から選んで実行してください。

起動後、メニューバーの **LLMs_Manager** で Gemini が有効になっていることを確認してください。


In [ ]:
#@title 起動方法 A: cloudflared（推奨）

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。cloudflared でトンネルを開きます...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI アクセス URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI

# CPUで動かす場合はこちら（GPU制限解除待ちの場合）
# !python main.py --cpu --dont-print-server --listen --enable-cors-header *

# もしT4 GPUが使えるようになったら、--cpu を外して以下にしてください
# GPUで実行する場合
!python main.py --dont-print-server --listen --enable-cors-header '*'

Selecting previously unselected package cloudflared.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...
/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[WARNING] WARNING: blake3 package not installed
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-05-25 10:22:38.016
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /c

In [ ]:
#@title 起動方法 B: localtunnel（cloudflared が使えない場合）

!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。localtunnel でトンネルを開きます...\n")
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("🔑 localtunnel パスワード / エンドポイント IP:", endpoint_ip)
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
#@title 起動方法 C: Colab iframe（WebSocket 非対応のため機能制限あり）

import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
    print("別ウィンドウで開く場合はこちら:")
    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server